# DataRetail 360 · Integración analítica confiable
## Evaluación Parcial 3 — IDY1102
**Estructuras y Visualización de Datos · Oscar Muñoz**  
**Entorno:** PostgreSQL de origen → Python/Pandas → SQL Server 2025.  
**Modalidad:** notebook técnico, ejecutable y documentado, con historial Git local.

Este trabajo integra las fuentes del caso para disponer información consistente de ventas,
clientes, productos, inventario y campañas. La empresa presenta cifras contradictorias porque
mezcla estados, impuestos, devoluciones, horarios y fuentes duplicadas. El objetivo de esta etapa
es construir la base analítica y demostrar calidad, integridad e historia temporal antes del consumo BI.

### Ruta de lectura y evaluación
| Criterio (20% cada uno) | Evidencia prevista |
|---|---|
| Modelo dimensional | Bus Matrix, diagrama, grano y diccionario |
| SCD | Tipos 1/2/3, justificación y pruebas temporales |
| Pipeline | Extracción, contratos, transformación, rechazos y logging |
| Almacenamiento | Bronze/Silver/Gold persistentes y SQL Server |
| Documentación | Decisiones, resultados, pruebas, linaje y Git |

### Ejecución
Crear el entorno con `python -m venv .venv`, instalar `requirements.txt`, seleccionar ese
intérprete en Jupyter/VS Code y completar `.env` a partir de `.env.example`.
Requiere PostgreSQL local poblado y SQL Server local con autenticación Windows y ODBC Driver 18.
Ejecutar las celdas en orden. La base `DataRetail360_DW` fue autorizada y creada previamente.
Los datos y logs se guardan en carpetas excluidas de Git. No se imprimen clientes ni credenciales.
El cuaderno es el entregable principal; el entorno, las bases y los archivos generados son sus dependencias.

## 1. Fuentes, alcance y decisiones
La fuente existente `idy1102_ep2` contiene 16 tablas `dr360_raw` que simulan los CSV,
Excel y sistemas del negocio. Se leen como tablas SQL; no se afirma haber recibido archivos
CSV/Excel independientes. Las cinco tablas `dr360_control` describen la simulación.
Las claves de persona real del generador **no se usan para resolver identidades**: hacerlo
convertiría una respuesta conocida de la simulación en una capacidad inexistente del negocio.

Se preservan las tablas de origen. La extracción se realiza en una transacción de solo lectura,
con una instantánea consistente. Las transformaciones ocurren en Pandas: es ETL hacia el DWH,
con conservación previa de crudos en Bronze. Medallion describe las capas de calidad, no obliga a usar Spark.

### Bus Matrix y granularidad
| Proceso | Una fila representa | Fecha | Producto | Sucursal | Cliente | Canal | Campaña |
|---|---|---|---|---|---|---|---|
| Ventas | Línea de documento + sistema POS/ECOM | Sí | Sí | Sí | Sí | Sí | Sí |
| Devoluciones | Registro de devolución | Sí | Sí | Sí | Sí | Sí | No |
| Inventario | Corte semanal × sucursal × producto | Sí | Sí | Sí | No | No | No |
| Campañas | Envío a un registro CRM | Sí | No | No | Sí | No | Sí |
| Web | Sesión | Sí | Producto principal | No | Sí | No | UTM |
| Reposición | Orden de reposición | Solicitud/recepción | Sí | Sí | No | No | No |

Las tablas de hechos no se unen directamente entre sí: se agregan al mismo grano antes de
comparar. El stock es semiaditivo: se suma entre productos/sucursales, **no entre cortes**.
La tabla de envíos no identifica un producto específico; no se inventa esa relación.
Los reportes legacy tienen grano agregado y quedan en una zona independiente de conciliación.

### Dimensiones y jerarquías
`dim_fecha`: día → mes → trimestre → año. `dim_producto`: producto → subcategoría → categoría
→ línea; marca es un atributo paralelo, no un nivel superior de categoría.
`dim_sucursal` y `dim_cliente`: comuna → región → país con roles geográficos distintos.
No se inventan provincia ni coordenadas que no existen en la fuente.
Se elige una constelación de estrellas con dimensiones conformadas; un copo de nieve separaría
geografías/categorías, pero añadiría joins sin necesidad para este volumen y uso académico.
Los miembros desconocidos usan clave 0; el canal digital sin local usa un miembro explícito.

### Contrato de indicadores
- Venta neta POS: documento EMITIDA, `monto_neto` sin IVA; se distribuye a líneas sin perder pesos.
- Venta neta e-commerce: ENTREGADO; mercadería `(subtotal-descuento)/1,19`, sin despacho,
  redondeada a CLP. La tasa 19% es un **supuesto del dataset**, no asesoría tributaria.
- Devoluciones: APROBADA y trazable al documento/producto; se descuentan una sola vez
  en fecha de devolución. Se explicita la base tributaria asumida de `monto_devuelto`.
- DEVUELTO/DEVOLUCION web no se reconoce como venta ENTREGADO. Sus devoluciones no se
  restan otra vez de ventas excluidas. Se informa esta limitación del modelo de estados de la fuente.
- Ticket promedio: venta antes de devoluciones / documentos distintos reconocidos.
- Cliente activo: persona resuelta con compra reconocida en los últimos seis meses respecto
  del corte del dataset; se excluye el cliente desconocido.
- Marketing: conversión reportada y retorno atribuido son indicadores de la plataforma;
  no se presentan como causalidad ni como ingreso conciliado con ventas.
- Inventario: riesgo de reposición según stock de sistema y mínimo; las diferencias con
  stock físico y antigüedad del conteo se reportan, no se corrigen inventando existencias.

Los errores de identidad se resuelven conservadoramente con RUT válido y correo único.
Los emparejamientos ambiguos quedan separados; similitud de nombres no basta para fusionar personas.

### Diagrama del modelo
La siguiente representación resume las relaciones. El archivo `modelo.drawio` contiene el
mismo diseño editable en diagrams.net. Cada hecho conserva su clave natural y las claves
sustitutas de las dimensiones aplicables.

```text
Fecha ─────┬── fact_ventas ───── Producto (SCD2)
Canal ─────┤       │                │
Campaña ───┤       ├── Sucursal ────┼── fact_inventario
Cliente ───┘       │                └── fact_reposicion
 (SCD2)            └── fact_devoluciones
   │
   ├── fact_marketing ── Campaña
   └── fact_web ──────── Producto / Campaña
```

SCD1 corrige atributos descriptivos; SCD2 conserva versiones de región/segmento del cliente
y categoría del producto; SCD3 mantiene zona actual/anterior de la sucursal.
La fuente solo contiene el estado actual: la primera carga **no reconstruye historia real**.
Se marca esa limitación y se demuestran cambios con casos controlados separados del negocio.